## 1. Dateset Exploration

In [1]:
from pathlib import Path

BDD10K_IMG_DIR = Path(r"D:\UCSD_courses\ECE285\project\data\bdd100k\10k\train")
BDD_OUT_DIR = Path(r"D:\UCSD_courses\ECE285\project\data\bdd_sampled")

bdd10k_names = {p.name for p in BDD10K_IMG_DIR.iterdir() if p.suffix.lower()==".jpg"}
print("bdd10k images:", len(bdd10k_names))

bdd10k images: 7000


### 1.1 Select Images in both BDD10K & Label

In [2]:
import json

BDD100K_LABEL_JSON = Path(r"D:\UCSD_courses\ECE285\project\data\bdd100k\labels\bdd100k_labels_images_train.json")

with BDD100K_LABEL_JSON.open("r", encoding="utf-8") as f:
    items = json.load(f)
print(f"number of labels: {len(items)}")
items_10k = [it for it in items if it.get("name") in bdd10k_names]
print("bdd100k label entries that exist in bdd10k:", len(items_10k))

number of labels: 69863
bdd100k label entries that exist in bdd10k: 2972


In [3]:
print(json.dumps(items_10k[:10], indent=4))

[
    {
        "name": "00054602-3bf57337.jpg",
        "attributes": {
            "weather": "clear",
            "scene": "city street",
            "timeofday": "daytime"
        },
        "timestamp": 10000,
        "labels": [
            {
                "category": "car",
                "attributes": {
                    "occluded": false,
                    "truncated": true,
                    "trafficLightColor": "none"
                },
                "manualShape": true,
                "manualAttributes": true,
                "box2d": {
                    "x1": 1254.239661,
                    "y1": 250.586088,
                    "x2": 1278.710959,
                    "y2": 273.099682
                },
                "id": 106
            },
            {
                "category": "car",
                "attributes": {
                    "occluded": true,
                    "truncated": false,
                    "trafficLightColor": "none"
             

In [4]:
import random
from collections import defaultdict

# ===== CONFIG =====
json_path = Path(r"..\data\bdd100k\labels\bdd100k_labels_images_train.json")
output_label_dir = Path(r"..\data\bdd_sampled\labels")
output_img_dir = Path(r"..\data\bdd_sampled\images")

num_samples = 500
seed = 42

# BDD100K commonly uses 1280x720
img_w = 1280
img_h = 720

# YOLO class id for "car"
class_id = 0

## 2. Data Processing

### 2.1 Check for avaliable 'weather' & 'timeofday'

In [5]:
domain_counter = defaultdict(int)
domain_items = defaultdict(list)
for item in items_10k:
    attr = item.get("attributes", {})
    weather = attr.get("weather", "undefined")
    tod = attr.get("timeofday", "undefined")
    domain_counter[(weather, tod)] += 1
    domain_items[(weather, tod)].append(item)

print("Number of domain combos:", len(domain_counter))
print("Available (weather, timeofday):")
for k in sorted(domain_counter.keys()):
    print(k, "->", domain_counter[k])

Number of domain combos: 21
Available (weather, timeofday):
('clear', 'dawn/dusk') -> 114
('clear', 'daytime') -> 829
('clear', 'night') -> 89
('foggy', 'dawn/dusk') -> 1
('foggy', 'daytime') -> 1
('overcast', 'dawn/dusk') -> 82
('overcast', 'daytime') -> 545
('partly cloudy', 'dawn/dusk') -> 45
('partly cloudy', 'daytime') -> 300
('partly cloudy', 'night') -> 1
('rainy', 'dawn/dusk') -> 26
('rainy', 'daytime') -> 176
('rainy', 'night') -> 12
('rainy', 'undefined') -> 1
('snowy', 'dawn/dusk') -> 29
('snowy', 'daytime') -> 202
('snowy', 'night') -> 11
('undefined', 'dawn/dusk') -> 27
('undefined', 'daytime') -> 473
('undefined', 'night') -> 2
('undefined', 'undefined') -> 6


### 2.2 Define some helper functions

In [6]:
SELECTED_DOMAINS = [
    ('clear', 'dawn/dusk'),
    ('clear', 'night'),
    ('overcast', 'dawn/dusk'),
    ('overcast', 'daytime'),
    ('partly cloudy', 'dawn/dusk'),
    ('rainy', 'dawn/dusk'),
    ('rainy', 'daytime'),
    ('rainy', 'night'),
    ('snowy', 'dawn/dusk'),
    ('snowy', 'daytime'),
    ('snowy', 'night'),
]


def bbox_area(box):
    return max(0.0, box["x2"] - box["x1"]) * max(0.0, box["y2"] - box["y1"])


def get_largest_car_box(item):
    """
    Return the largest 'car' box2d in this item.
    If no car exists, return None.
    """
    car_boxes = []

    for label in item.get("labels", []):
        if label.get("category") == "car" and "box2d" in label:
            car_boxes.append(label["box2d"])

    if not car_boxes:
        return None

    return max(car_boxes, key=bbox_area)


def xyxy_to_yolo(box, img_w, img_h):
    """
    Convert box from (x1, y1, x2, y2) to YOLO format:
    (x_center, y_center, width, height), normalized to [0,1]
    """
    x1, y1, x2, y2 = box["x1"], box["y1"], box["x2"], box["y2"]

    xc = ((x1 + x2) / 2.0) / img_w
    yc = ((y1 + y2) / 2.0) / img_h
    w  = (x2 - x1) / img_w
    h  = (y2 - y1) / img_h

    return xc, yc, w, h

def sample_selected(domain_dict, n=500):
    # sample images from selected domain
    domain_samples = []
    for domain in SELECTED_DOMAINS:
        domain_samples.extend(domain_dict[domain])
    # 500 samples
    samples = random.choices(domain_samples, k=n)
    return samples

### 2.3 Extract selected images and labels

In [7]:

import shutil

samples = sample_selected(domain_items)
print(f"Len of Samples: {len(samples)}")

# extract largest bbox for each image
for s in samples:
    # get the bbox
    bbox = get_largest_car_box(s)
    if bbox != None:
        # convert to YOLO
        xc, yc, w, h = xyxy_to_yolo(bbox, img_w, img_h)
        # generate label
        img_name = s['name']
        label_name = Path(img_name).stem + '.txt'
        output_name = output_label_dir / label_name
        with open(output_name, 'w+') as labelf:
            labelf.write(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")

        # copy the selected image
        src_img = BDD10K_IMG_DIR / img_name
        dst_img = output_img_dir / img_name
        shutil.copy2(src_img, dst_img)

Len of Samples: 500
